# BeltCrack9kd: single-frame YOLO crack detector

Trains on the compact upload built by `vision/prepare_dataset.py` (`beltcrack_yolo960.zip`), chooses a confidence
threshold on **val only**, freezes, then scores **test exactly once**. Exports ONNX for the demo.

**Before running (Colab):** put `beltcrack_yolo960.zip` in a Google Drive folder called `BeltCrack`, then
`Runtime > Change runtime type > T4 GPU` and `Runtime > Run all`. If the session disconnects, `Run all` again:
training resumes, and finished steps are skipped (the test step refuses to run twice).

Everything to bring back to the repo ends up in `BeltCrack/artifacts/` on Drive.

## 1. Settings

In [ ]:
ZIP_NAME   = "beltcrack_yolo960.zip"
MODEL      = "yolo11s.pt"
IMGSZ      = 640          # first run; a second run at 960 only if val mAP50 lands well below the 20-30 guide
EPOCHS     = 100
PATIENCE   = 15
BATCH      = 32           # T4 at 640. Lower it on "CUDA out of memory"
SEED       = 0
CACHE      = "ram"        # caches decoded images; speeds up Colab's 2-CPU data loading, does not change training

PAPER_CONF, PAPER_IOU = 0.001, 0.65     # the paper's evaluation settings (arXiv 2506.17892)
TARGET_P, MIN_R       = 0.90, 0.60      # threshold rule: P >= 0.9 if R there >= 0.6, else max F1 (val only)
STRIDE                = (15, 30)        # frames advanced per simulated lap (same as the simulator's capture stride)
RUN_NAME   = f"beltcrack9kd_{MODEL.split('.')[0]}_{IMGSZ}"

## 2. Install and check the GPU

In [ ]:
%pip install -q ultralytics onnx onnxslim onnxruntime

import torch
import ultralytics

print("torch", torch.__version__, "| ultralytics", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a T4 GPU")

## 3. Paths, copy the zip to local disk and unzip

In [ ]:
import hashlib
import json
import shutil
import sys
import zipfile
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/BeltCrack")
    WORK = Path("/content")
else:                                   # local or Kaggle: a BeltCrack folder next to the notebook
    DRIVE = Path("BeltCrack").resolve()
    WORK = Path.cwd()

RUNS = DRIVE / "runs"                   # checkpoints on Drive survive disconnects
ART = DRIVE / "artifacts"
RUNS.mkdir(parents=True, exist_ok=True)
ART.mkdir(parents=True, exist_ok=True)
DATA = WORK / "beltcrack_yolo960"

if not (DATA / "data.yaml").exists():
    src = DRIVE / ZIP_NAME
    assert src.exists(), f"Put {ZIP_NAME} in {DRIVE}"
    local = WORK / ZIP_NAME
    if not local.exists():
        shutil.copy(src, local)
    with zipfile.ZipFile(local) as z:
        z.extractall(WORK)

DATA_YAML = WORK / "data.yaml"          # same content, with an absolute path so Ultralytics finds the folders
DATA_YAML.write_text(f"path: {DATA}\n" + (DATA / "data.yaml").read_text())
manifest = json.loads((DATA / "manifest.json").read_text())
print(json.dumps(manifest["counts"], indent=1), "\nclean_val tiles:", manifest["clean_val"]["tiles"])
for s in ("train", "val", "test"):
    n = len(list((DATA / "images" / s).glob("*.jpg")))
    assert n == manifest["counts"][s]["images"], (s, n)

## 4. Train (Ultralytics defaults; resumes after a disconnect)

In [ ]:
from ultralytics import YOLO

RUN_DIR = RUNS / RUN_NAME
LAST, BEST = RUN_DIR / "weights" / "last.pt", RUN_DIR / "weights" / "best.pt"

def finished(ckpt):
    return torch.load(ckpt, map_location="cpu", weights_only=False).get("epoch", -1) == -1

if LAST.exists() and finished(LAST):
    print("Training already finished:", RUN_DIR)
elif LAST.exists():
    YOLO(str(LAST)).train(resume=True)
else:
    YOLO(MODEL).train(data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
                      seed=SEED, cache=CACHE, project=str(RUNS), name=RUN_NAME, exist_ok=True)

## 5. Val: mAP at the paper's settings, the precision-recall curve, and the threshold

Only val is used for decisions. The 20-30 mAP50 guide comes from the paper's 9kd test rows at 512 px; it is a
rough guide, not a gate. Much higher: re-check leakage. Much lower: re-check the conversion first.

In [ ]:
import csv

import numpy as np

model = YOLO(str(BEST))
val = model.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, conf=PAPER_CONF, iou=PAPER_IOU, batch=BATCH,
                plots=True, project=str(RUNS), name=RUN_NAME + "_val", exist_ok=True)
print(f"val mAP50 {val.box.map50:.4f} | mAP50-95 {val.box.map:.4f}")

# Ultralytics keeps P and R against confidence (matched at IoU 0.5): curves_results = [PR, F1, P, R]
conf_grid = np.asarray(val.box.curves_results[2][0])
P = np.asarray(val.box.curves_results[2][1])[0]
R = np.asarray(val.box.curves_results[3][1])[0]
F1 = 2 * P * R / np.maximum(P + R, 1e-9)

with open(ART / "pr_curve_val.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["conf", "precision", "recall", "f1"])
    w.writerows(zip(conf_grid.round(4), P.round(4), R.round(4), F1.round(4)))

ok = np.where(P >= TARGET_P)[0]
cand_p = None if len(ok) == 0 else int(ok[0])               # lowest conf reaching P >= 0.9
cand_f = int(F1.argmax())
use_p = cand_p is not None and R[cand_p] >= MIN_R
i = cand_p if use_p else cand_f
THRESHOLD = float(conf_grid[i])
choice = {
    "rule": f"P >= {TARGET_P} if R there >= {MIN_R}, else max F1 (val frames)",
    "chosen": "precision>=0.9" if use_p else "max_f1", "threshold": round(THRESHOLD, 4),
    "at_threshold": {"precision": round(float(P[i]), 4), "recall": round(float(R[i]), 4), "f1": round(float(F1[i]), 4)},
    "candidate_precision_0.9": None if cand_p is None else {"conf": round(float(conf_grid[cand_p]), 4),
        "precision": round(float(P[cand_p]), 4), "recall": round(float(R[cand_p]), 4), "f1": round(float(F1[cand_p]), 4)},
    "candidate_max_f1": {"conf": round(float(conf_grid[cand_f]), 4), "precision": round(float(P[cand_f]), 4),
        "recall": round(float(R[cand_f]), 4), "f1": round(float(F1[cand_f]), 4)},
}
print(json.dumps(choice, indent=1))

## 6. Before freezing (val only, reported, not used to choose)

- **False detections per frame** on the box-free val tiles (`clean_val/`; the dataset has no crack-free frames).
- **Confirmation effect:** step through each val sequence with a seeded stride of 15-30 frames per "lap", as the
  simulator does. A location is confirmed once it is detected on 2 laps. For crack sequences we count laps with a
  detection and boxes matching no label (IoU < 0.5); for box-free tile "locations" every hit is false, and we count
  how many single-lap false hits never reach confirmation.

In [ ]:
import random
import re
from collections import defaultdict


def natural(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]

def predict(paths, conf):
    out = []
    for k in range(0, len(paths), 32):
        out += model.predict([str(p) for p in paths[k:k + 32]], imgsz=IMGSZ, conf=conf, verbose=False)
    return out

# ---- false hits per frame on clean tiles
tiles = sorted((DATA / "clean_val" / "images").glob("*.jpg"), key=lambda p: natural(p.name))
tile_res = predict(tiles, THRESHOLD)
tile_hits = [len(r.boxes) for r in tile_res]
clean = {"source": "box-free val tiles (no crack-free frames in the dataset)", "tiles": len(tiles),
         "false_detections_per_frame": round(float(np.mean(tile_hits)), 4),
         "tiles_with_any_detection": int(sum(h > 0 for h in tile_hits))}
print(clean)

# ---- confirmation effect
def iou(a, b):
    x1, y1 = np.maximum(a[:2], b[:2])
    x2, y2 = np.minimum(a[2:], b[2:])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def labels_xyxy(img_path, w, h):
    rows = (DATA / "labels" / "val" / (img_path.stem + ".txt")).read_text().split()
    a = np.array(rows, dtype=float).reshape(-1, 5)[:, 1:]
    return np.stack([(a[:, 0] - a[:, 2] / 2) * w, (a[:, 1] - a[:, 3] / 2) * h,
                     (a[:, 0] + a[:, 2] / 2) * w, (a[:, 1] + a[:, 3] / 2) * h], 1)

rng = random.Random(SEED)
seqs = defaultdict(list)
for p in sorted((DATA / "images" / "val").glob("*.jpg"), key=lambda p: natural(p.name)):
    seqs[p.stem.split("__")[0]].append(p)
crack_locs = []
for seq, frames in sorted(seqs.items(), key=lambda kv: natural(kv[0])):
    idx, laps = 0, []
    while idx < len(frames):
        laps.append(frames[idx])
        idx += rng.randint(*STRIDE)
    res = predict(laps, THRESHOLD)
    detected, false_boxes, boxes = 0, 0, 0
    for p, r in zip(laps, res):
        b = r.boxes.xyxy.cpu().numpy()
        h, w = r.orig_shape
        gt = labels_xyxy(p, w, h)
        detected += len(b) > 0
        boxes += len(b)
        false_boxes += sum(1 for x in b if max((iou(x, g) for g in gt), default=0) < 0.5)
    crack_locs.append({"sequence": seq, "laps": len(laps), "laps_detected": int(detected),
                       "confirmed": detected >= 2, "boxes": int(boxes), "boxes_matching_no_label": int(false_boxes)})

tile_seqs = defaultdict(list)
for p, h in zip(tiles, tile_hits):
    if p.stem.endswith("tile0"):
        tile_seqs[p.stem.split("__")[0]].append(h)
clean_locs = []
for seq, hits in sorted(tile_seqs.items(), key=lambda kv: natural(kv[0])):
    idx, lap_hits = 0, []
    while idx < len(hits):                    # tiles come from every 10th frame, so a 15-30 frame stride is 2-3 tiles
        lap_hits.append(hits[idx] > 0)
        idx += max(1, round(rng.randint(*STRIDE) / 10))
    clean_locs.append({"sequence": seq, "laps": len(lap_hits), "laps_with_false_hit": int(sum(lap_hits)),
                       "confirmed_false_location": sum(lap_hits) >= 2})
single = sum(c["laps_with_false_hit"] for c in clean_locs)
removed = sum(c["laps_with_false_hit"] for c in clean_locs if not c["confirmed_false_location"])
confirmation = {
    "stride_frames": list(STRIDE), "crack_locations": crack_locs,
    "crack_locations_confirmed": f"{sum(c['confirmed'] for c in crack_locs)} of {len(crack_locs)}",
    "clean_tile_locations": clean_locs,
    "single_lap_false_hits_on_clean_locations": single, "removed_by_2_lap_confirmation": removed,
    "clean_locations_confirmed_false": sum(c["confirmed_false_location"] for c in clean_locs),
    "note": "frames drawn independently make confirmation look kinder than persistent real stains would",
}
print(json.dumps({k: v for k, v in confirmation.items() if k not in ("crack_locations", "clean_tile_locations")}, indent=1))

## 7. Freeze (writes `freeze.json` and the val part of `vision_eval.json`)

In [ ]:
FREEZE = ART / "freeze.json"
EVAL = ART / "vision_eval.json"
sha = hashlib.sha256(BEST.read_bytes()).hexdigest()
if FREEZE.exists():
    frozen = json.loads(FREEZE.read_text())
    assert frozen["best_pt_sha256"] == sha, "best.pt changed after the freeze"
    THRESHOLD = frozen["threshold"]
    print("already frozen:", frozen)
else:
    frozen = {"threshold": round(THRESHOLD, 4), "imgsz": IMGSZ, "model": MODEL, "best_pt_sha256": sha,
              "ultralytics": ultralytics.__version__, "torch": torch.__version__}
    FREEZE.write_text(json.dumps(frozen, indent=1))
    EVAL.write_text(json.dumps({
        "dataset": manifest["dataset"], "counts": manifest["counts"], "train": {"model": MODEL, "imgsz": IMGSZ,
            "epochs_max": EPOCHS, "patience": PATIENCE, "batch": BATCH, "seed": SEED, "defaults": "Ultralytics"},
        "val": {"map50": round(float(val.box.map50), 4), "map50_95": round(float(val.box.map), 4),
                "eval_conf": PAPER_CONF, "eval_iou": PAPER_IOU},
        "threshold_choice": choice, "false_hits_clean_val": clean, "confirmation_effect_val": confirmation,
        "freeze": frozen}, indent=1))
    print("frozen:", frozen)
shutil.copy(RUN_DIR / "results.csv", ART / "results.csv")

## 8. Test, exactly once

In [ ]:
ev = json.loads(EVAL.read_text())
if "test" in ev:
    print("Test was already scored; not running it again:", ev["test"])
else:
    assert json.loads(FREEZE.read_text())["best_pt_sha256"] == hashlib.sha256(BEST.read_bytes()).hexdigest()
    t = YOLO(str(BEST)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, conf=PAPER_CONF, iou=PAPER_IOU,
                            batch=BATCH, plots=True, project=str(RUNS), name=RUN_NAME + "_test", exist_ok=True)
    cg = np.asarray(t.box.curves_results[2][0])
    tp = float(np.interp(THRESHOLD, cg, np.asarray(t.box.curves_results[2][1])[0]))
    tr = float(np.interp(THRESHOLD, cg, np.asarray(t.box.curves_results[3][1])[0]))
    ev["test"] = {"map50": round(float(t.box.map50), 4), "map50_95": round(float(t.box.map), 4),
                  "eval_conf": PAPER_CONF, "eval_iou": PAPER_IOU, "threshold": THRESHOLD,
                  "precision_at_threshold": round(tp, 4), "recall_at_threshold": round(tr, 4),
                  "f1_at_threshold": round(2 * tp * tr / max(tp + tr, 1e-9), 4)}
    EVAL.write_text(json.dumps(ev, indent=1))
    print(json.dumps(ev["test"], indent=1))

## 9. Export ONNX and copy the weights

The ONNX/`.pt` parity check runs locally after this, on the demo frames, with the same `best.pt`.

In [ ]:
onnx_path = Path(YOLO(str(BEST)).export(format="onnx", imgsz=IMGSZ, opset=12, dynamic=False, simplify=True))
shutil.copy(onnx_path, ART / "best.onnx")
shutil.copy(BEST, ART / "best.pt")
print(sorted(p.name for p in ART.iterdir()))
print("Download the BeltCrack/artifacts folder from Drive into vision/artifacts/ in the repo.")